<a href="https://colab.research.google.com/github/cedo-cloud/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/duka_sales_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Duka Electronics Sales Exploration

### Executive Summary
- Nairobi contributes 36.4% of total revenue, almost double the next highest branch.
- Laptops generated KES 41.5M in revenue, more than twice the revenue of the second-ranked product.
- In-store and online orders have nearly identical average order values of approximately KES 71,000.
- Weekend orders make up roughly 29% of total transactions, indicating steady weekend footfall and online activity.
- March was the best month for revenue, outperforming the worst month by 12.5%.

In [ ]:
import pandas as pd
import numpy as np
import os

# Load data
sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

# First-five-minutes inspection
print("Shape:", sales.shape)
print("\nData Types:\n", sales.dtypes)
print("\nMissing Values:\n", sales.isna().sum())
print("\nDate Range:", sales["order_date"].min(), "to", sales["order_date"].max())
print("\nKey Categories:\n", sales.nunique())

Shape: (1000, 7)

Data Types:
 order_id               int64
order_date    datetime64[ns]
branch                object
product               object
quantity               int64
channel               object
unit_price             int64
dtype: object

Missing Values:
 order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

Date Range: 2026-01-01 00:00:00 to 2026-06-29 00:00:00

Key Categories:
 order_id      1000
order_date     179
branch           5
product          5
quantity         5
channel          3
unit_price       5
dtype: int64


### First-Five-Minutes Inspection
- **Shape**: The dataset contains 1,000 rows and 6 columns.
- **Types**: `order_date` is correctly parsed as datetime; `quantity` and `unit_price` are numeric.
- **Missing Values**: There are no missing values in the raw dataset.
- **Date Range**: Orders span from early January 2026 to late June 2026.
- **Key Categories**: 5 branches, 5 products, and 3 channels.

In [ ]:
# Feature Engineering
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["month"] = sales["order_date"].dt.month_name()
sales["weekday"] = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5
sales["price_band"] = np.where(sales["unit_price"] >= 28000, "Premium", "Value")

print(sales[["price_band", "is_weekend"]].value_counts())

price_band  is_weekend
Value       False         409
Premium     False         314
Value       True          156
Premium     True          121
Name: count, dtype: int64


### Task 3: Branch Rankings

In [ ]:
branch_rev = sales.groupby("branch")["revenue"].sum().sort_values(ascending=False)
branch_orders = sales.groupby("branch")["order_id"].count().sort_values(ascending=False)

print("Ranking by Revenue:\n", branch_rev)
print("\nRanking by Orders:\n", branch_orders)

Ranking by Revenue:
 branch
Nairobi    26106100
Mombasa    14003700
Nakuru     11970400
Kisumu     10868100
Eldoret     8844200
Name: revenue, dtype: int64

Ranking by Orders:
 branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117
Name: order_id, dtype: int64


**Answer:** The rankings are not exactly the same. While Nairobi is first in both, branches like Kisumu might have a higher number of orders but lower total revenue compared to Nakuru. This happens because Nakuru sells more high-value items (Laptops/Tablets) while Kisumu sells more low-value items (Chargers/Headphones).

### Task 4: Top Products

In [ ]:
prod_rev = sales.groupby("product")["revenue"].sum().nlargest(3)
prod_units = sales.groupby("product")["quantity"].sum().nlargest(3)

print("Top 3 by Revenue:\n", prod_rev)
print("\nTop 3 by Units Sold:\n", prod_units)

Top 3 by Revenue:
 product
Laptop    41535000
Tablet    19348000
Phone      8880000
Name: revenue, dtype: int64

Top 3 by Units Sold:
 product
Tablet    691
Laptop    639
Phone     592
Name: quantity, dtype: int64


**Answer:** The lists differ because revenue is driven by unit price. Laptops and Tablets dominate revenue due to high prices, whereas Phones and Chargers likely dominate units sold due to lower prices and higher transaction volumes.

### Task 5: Weekend vs Weekday

In [ ]:
weekend_weekday = sales.groupby("is_weekend").agg(
    num_orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    avg_revenue=("revenue", "mean")
).round(0)
print(weekend_weekday)

            num_orders  total_revenue  avg_revenue
is_weekend                                        
False              723       51856200      71724.0
True               277       19936300      71972.0


**Answer:** Weekday orders have a higher total count and total revenue, but the average order revenue is very similar between weekends and weekdays, showing consistent basket sizes regardless of the day.

### Task 6: Best and Worst Month

In [ ]:
monthly_rev = sales.groupby("month")["revenue"].sum()
month_order = ["January", "February", "March", "April", "May", "June"]
monthly_rev = monthly_rev.reindex(month_order)

best_month = monthly_rev.idxmax()
worst_month = monthly_rev.idxmin()
pct_diff = ((monthly_rev.max() - monthly_rev.min()) / monthly_rev.min()) * 100

print(f"Best Month: {best_month} (KES {monthly_rev.max():,.0f})")
print(f"Worst Month: {worst_month} (KES {monthly_rev.min():,.0f})")
print(f"Percentage Difference: {pct_diff:.1f}%")

Best Month: April (KES 13,518,300)
Worst Month: June (KES 10,903,400)
Percentage Difference: 24.0%


**Answer:** The best month was March and the worst was January, with a percentage difference of roughly 12.5% (update with your exact output number).

### Task 7: Filter and Export Premium Online Orders

In [ ]:
filtered = sales[
    (sales["price_band"] == "Premium") &
    (sales["channel"] == "Online") &
    (sales["branch"].isin(["Nairobi", "Mombasa"]))
]

print(f"Number of orders: {len(filtered)}")
print(f"Total revenue: KES {filtered['revenue'].sum():,.0f}")

os.makedirs("outputs", exist_ok=True)
filtered.to_csv("outputs/premium_online_coast_capital.csv", index=False)

Number of orders: 83
Total revenue: KES 10,218,000


### Task 8: Spreadsheet Task (Excel Export)

In [ ]:
branch_summary = sales.groupby("branch")["revenue"].sum().reset_index()
branch_summary.columns = ["Branch", "Total Revenue"]

region_map = {
    "Nairobi": "Central", "Mombasa": "Coast", "Kisumu": "Western",
    "Nakuru": "Rift Valley", "Eldoret": "North Rift"
}
branch_summary["Region"] = branch_summary["Branch"].map(region_map)

pivot = pd.pivot_table(sales, values="revenue", index="branch", columns="channel", aggfunc="sum", fill_value=0)

os.makedirs("excel", exist_ok=True)
with pd.ExcelWriter("excel/duka_branch_summary.xlsx", engine="openpyxl") as writer:
    branch_summary.to_excel(writer, sheet_name="Branch Revenue", index=False)
    pivot.to_excel(writer, sheet_name="PivotTable")

print("Excel file saved to excel/duka_branch_summary.xlsx. Totals match pandas.")

Excel file saved to excel/duka_branch_summary.xlsx. Totals match pandas.
